In [13]:
# Установка необходимых библиотек
!pip install torch torchvision scikit-learn tqdm numpy kaggle timm -q

In [14]:
# Импорт библиотек для работы с данными, обучением моделей и оценкой качества
import os
import shutil
import random
import numpy as np
import json
import torch
import copy
from tqdm import tqdm
from collections import Counter
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score

In [15]:
# Фиксация параметра seed для воспроизводимости результатов
SEED = 41

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [16]:
# Настройка доступа к Kaggle API
# === НЕОБХОДИМО ВВЕСТИ СВОИ ДАННЫЕ С KAGGLE ===
!mkdir -p /root/.kaggle

kaggle_token = {
    "username": "USERNAME",
    "key": "API_TOKEN"
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_token, f)

In [17]:
# Настройка прав доступа и загрузка датасета с Kaggle
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d jpjp0902/car-classification -q
!unzip -q car-classification.zip

Dataset URL: https://www.kaggle.com/datasets/jpjp0902/car-classification
License(s): unknown


In [18]:
# Анализ структуры датасета
root = "/content/car_ori"

classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])

print("Количество папок-классов:", len(classes))
print("Первые 10 классов:", classes[:10])

ext_counter = Counter()
class_counts_before = {}

for cls in classes:
    cls_path = os.path.join(root, cls)
    files = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]

    class_counts_before[cls] = len(files)

    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_counter[ext] += 1

print("Форматы файлов во всем датасете:")
for ext, cnt in sorted(ext_counter.items()):
    print(f"{ext}: {cnt}")

print("\nПримеры количества файлов по классам:")
for cls in classes[:10]:
    print(cls, "->", class_counts_before[cls])

    from collections import defaultdict

duplicates_by_class = {}

for cls in classes:
    cls_path = os.path.join(root, cls)
    base_names = defaultdict(list)

    for f in os.listdir(cls_path):
        full_path = os.path.join(cls_path, f)
        if os.path.isfile(full_path):
            base_name = os.path.splitext(f)[0]
            base_names[base_name].append(f)

    dupes = {k: v for k, v in base_names.items() if len(v) > 1}
    if dupes:
        duplicates_by_class[cls] = dupes

print("Классов с дублями по имени:", len(duplicates_by_class))

for cls in list(duplicates_by_class.keys())[:5]:
    print(f"\nКласс: {cls}")
    for k, v in list(duplicates_by_class[cls].items())[:5]:
        print(" ", k, "->", v)

Количество папок-классов: 33
Первые 10 классов: ['AVANTE', 'CASPER', 'EV6', 'G70', 'GRANDEUR', 'GV60', 'IONIQ5', 'IONIQ6', 'K5', 'K8']
Форматы файлов во всем датасете:
.jpeg: 140
.jpg: 8921
.png: 839

Примеры количества файлов по классам:
AVANTE -> 300
CASPER -> 300
EV6 -> 300
G70 -> 300
GRANDEUR -> 300
GV60 -> 300
IONIQ5 -> 300
IONIQ6 -> 300
K5 -> 300
K8 -> 300
Классов с дублями по имени: 17

Класс: AVANTE
  AVANTE (3) -> ['AVANTE (3).png', 'AVANTE (3).jpg']
  AVANTE (58) -> ['AVANTE (58).png', 'AVANTE (58).jpg']
  AVANTE (53) -> ['AVANTE (53).jpg', 'AVANTE (53).png']
  AVANTE (52) -> ['AVANTE (52).png', 'AVANTE (52).jpg']
  AVANTE (18) -> ['AVANTE (18).png', 'AVANTE (18).jpg']

Класс: CASPER
  CASPER (15) -> ['CASPER (15).jpg', 'CASPER (15).png']
  CASPER (22) -> ['CASPER (22).jpg', 'CASPER (22).png']
  CASPER (50) -> ['CASPER (50).jpg', 'CASPER (50).png']
  CASPER (42) -> ['CASPER (42).png', 'CASPER (42).jpg']
  CASPER (18) -> ['CASPER (18).jpg', 'CASPER (18).png']

Класс: GRANDEUR


In [19]:
# Очистка датасета (оставляем только .jpg файлы)
removed = 0

for cls in classes:
    cls_path = os.path.join(root, cls)

    for f in os.listdir(cls_path):
        if not f.lower().endswith(".jpg"):
            os.remove(os.path.join(cls_path, f))
            removed += 1

print("Удалено файлов:", removed)

Удалено файлов: 979


In [20]:
# Проверка датасета после очистки
ext_counter = Counter()
class_counts = {}

for cls in classes:
    cls_path = os.path.join(root, cls)
    files = os.listdir(cls_path)

    class_counts[cls] = len(files)

    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_counter[ext] += 1

print("Форматы после очистки:", ext_counter)

print("\nПримеры классов:")
for cls in classes[:10]:
    print(cls, "->", class_counts[cls])

Форматы после очистки: Counter({'.jpg': 8921})

Примеры классов:
AVANTE -> 240
CASPER -> 240
EV6 -> 300
G70 -> 300
GRANDEUR -> 240
GV60 -> 300
IONIQ5 -> 240
IONIQ6 -> 180
K5 -> 300
K8 -> 300


In [ ]:
# Разделение датасета на train / val / test
SEED = 41
random.seed(SEED)

source_dir = "/content/car_ori"
base_dir = "/content/dataset"

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

for folder in [train_dir, val_dir, test_dir]:
    os.makedirs(folder, exist_ok=True)

train_ratio = 0.7
val_ratio = 0.15

classes = sorted(os.listdir(source_dir))

for cls in tqdm(classes):
    cls_path = os.path.join(source_dir, cls)
    images = sorted([img for img in os.listdir(cls_path) if img.endswith(".jpg")])

    random.shuffle(images)

    train_split = int(len(images) * train_ratio)
    val_split = int(len(images) * (train_ratio + val_ratio))

    train_imgs = images[:train_split]
    val_imgs = images[train_split:val_split]
    test_imgs = images[val_split:]

    for folder, imgs in zip(
        [train_dir, val_dir, test_dir],
        [train_imgs, val_imgs, test_imgs]
    ):
        cls_folder = os.path.join(folder, cls)
        os.makedirs(cls_folder, exist_ok=True)

        for img in imgs:
            shutil.copy(
                os.path.join(cls_path, img),
                os.path.join(cls_folder, img)
            )

100%|██████████| 33/33 [00:00<00:00, 33.62it/s]


In [ ]:
# Преобразования изображений: изменение размера и нормализация пикселей
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Подготовка датасетов и загрузчиков данных
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)
val_dataset = datasets.ImageFolder("/content/dataset/val", transform=val_transform)
test_dataset = datasets.ImageFolder("/content/dataset/test", transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=2, pin_memory=True)

num_classes = len(train_dataset.classes)
print("Классов:", num_classes)

Классов: 33


In [ ]:
# Использование предобученной ResNet18: заморозка слоев и обучение нового классификатора
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 97.6MB/s]


In [ ]:
# Обучение модели с оценкой на валидационной выборке
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.4797 | Val Loss: 1.7785 | Val Acc: 0.6007 | Val F1-macro: 0.5875
Epoch 2 | Train Loss: 1.5438 | Val Loss: 1.4187 | Val Acc: 0.6351 | Val F1-macro: 0.6441
Epoch 3 | Train Loss: 1.2355 | Val Loss: 1.1646 | Val Acc: 0.7097 | Val F1-macro: 0.7061
Epoch 4 | Train Loss: 1.0784 | Val Loss: 1.1037 | Val Acc: 0.7097 | Val F1-macro: 0.7092
Epoch 5 | Train Loss: 0.9565 | Val Loss: 1.0251 | Val Acc: 0.7388 | Val F1-macro: 0.7376


In [ ]:
# Использование предобученной ViT (Vision Transformer): заморозка слоев и обучение классификатора
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.heads.head.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 69.6MB/s]


In [ ]:
# Обучение ViT с оценкой качества на валидационной выборке
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.8982 | Val Loss: 1.2191 | Val Acc: 0.7313 | Val F1-macro: 0.7283
Epoch 2 | Train Loss: 0.9659 | Val Loss: 0.8907 | Val Acc: 0.7843 | Val F1-macro: 0.7838
Epoch 3 | Train Loss: 0.6953 | Val Loss: 0.7436 | Val Acc: 0.8201 | Val F1-macro: 0.8202
Epoch 4 | Train Loss: 0.5456 | Val Loss: 0.6566 | Val Acc: 0.8276 | Val F1-macro: 0.8275
Epoch 5 | Train Loss: 0.4514 | Val Loss: 0.5871 | Val Acc: 0.8567 | Val F1-macro: 0.8554


In [ ]:
# В ходе экспериментов было установлено, что трансформерная модель (ViT)
# показала более высокие значения метрик качества по сравнению со сверточной
# моделью (ResNet18) на данном датасете

In [ ]:
# Оценка модели на тестовой выборке (бейзлайн - ResNet)
model.eval()
correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.7149 | Test F1-macro: 0.7142


In [ ]:
# Оценка модели на тестовой выборке (бейзлайн - ViT)
model.eval()
correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.8358 | Test F1-macro: 0.8366


In [ ]:
# Улучшение бейзлайна
# В ходе работы были выдвинуты следующие гипотезы улучшения baseline-модели:

# 1. Подбор более эффективной модели
# Предполагается, что выбор другой модели (в случае с каждой архитектурой) может повысить качество классификации.

# 2. Применение fine-tuning предобученной модели
# Предполагается, что размораживание слоев модели позволит адаптировать извлекаемые признаки под специфику датасета и повысить качество классификации
# по сравнению с обучением только классификационного слоя.

# 3. Подбор скорости обучения (learning rate)
# Предполагается, что выбор более подходящего значения learning rate позволит ускорить сходимость модели
# и получить более высокие значения метрик качества.

# 4. Использование аугментации изображений
# Предполагается, что применение преобразований обучающих данных повысит разнообразие обучающей выборки и улучшит обобщающую способность модели.

# 5. Подбор размера батча (batch size)
# Размер батча определяет, сколько изображений обрабатывается за один шаг обучения.
# Предполагается, что изменение этого параметра может улучшить итоговое качество модели.

# 6. Подбор количества эпох обучения
# Предполагается, что увеличение или уточнение числа эпох позволит модели полнее извлечь закономерности из обучающей выборки. При этом важно
# подобрать такое значение, при котором метрики качества достигают пиковых значений и переобучение не начинает заметно проявляться.

In [ ]:
# 1. Инициализация ResNet50 и обучение
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

model.to(device)

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.4344 | Val Loss: 1.9432 | Val Acc: 0.6672 | Val F1-macro: 0.6659
Epoch 2 | Train Loss: 1.4163 | Val Loss: 1.4484 | Val Acc: 0.7157 | Val F1-macro: 0.7173
Epoch 3 | Train Loss: 1.0512 | Val Loss: 1.1138 | Val Acc: 0.7373 | Val F1-macro: 0.7362
Epoch 4 | Train Loss: 0.8468 | Val Loss: 1.0030 | Val Acc: 0.7701 | Val F1-macro: 0.7690
Epoch 5 | Train Loss: 0.7137 | Val Loss: 0.9095 | Val Acc: 0.7754 | Val F1-macro: 0.7727


In [ ]:
# Инициализация EfficientNet-B0 и обучение
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model.to(device)

optimizer = optim.Adam(model.classifier[1].parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 86.5MB/s]


Epoch 1 | Train Loss: 2.2793 | Val Loss: 1.5390 | Val Acc: 0.6507 | Val F1-macro: 0.6473
Epoch 2 | Train Loss: 1.3899 | Val Loss: 1.1853 | Val Acc: 0.7224 | Val F1-macro: 0.7193
Epoch 3 | Train Loss: 1.1312 | Val Loss: 1.0725 | Val Acc: 0.7291 | Val F1-macro: 0.7246
Epoch 4 | Train Loss: 0.9783 | Val Loss: 0.9957 | Val Acc: 0.7381 | Val F1-macro: 0.7338
Epoch 5 | Train Loss: 0.8706 | Val Loss: 0.9481 | Val Acc: 0.7500 | Val F1-macro: 0.7478


In [ ]:
# Инициализация DenseNet-121 и обучение
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Linear(model.classifier.in_features, num_classes)

model.to(device)

optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 123MB/s]


Epoch 1 | Train Loss: 2.3931 | Val Loss: 1.7034 | Val Acc: 0.6060 | Val F1-macro: 0.6145
Epoch 2 | Train Loss: 1.4192 | Val Loss: 1.2715 | Val Acc: 0.6933 | Val F1-macro: 0.6921
Epoch 3 | Train Loss: 1.1104 | Val Loss: 1.0914 | Val Acc: 0.7261 | Val F1-macro: 0.7264
Epoch 4 | Train Loss: 0.9146 | Val Loss: 0.9867 | Val Acc: 0.7388 | Val F1-macro: 0.7413
Epoch 5 | Train Loss: 0.8080 | Val Loss: 0.9067 | Val Acc: 0.7500 | Val F1-macro: 0.7514


In [ ]:
# Инициализация предобученной ConvNeXt и замена классификатора
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

optimizer = optim.Adam(model.classifier[2].parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:01<00:00, 59.5MB/s]


In [ ]:
# Обучение модели ConvNeXt
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.1624 | Val Loss: 1.4063 | Val Acc: 0.7007 | Val F1-macro: 0.7035
Epoch 2 | Train Loss: 1.1905 | Val Loss: 0.9902 | Val Acc: 0.7776 | Val F1-macro: 0.7758
Epoch 3 | Train Loss: 0.8822 | Val Loss: 0.8285 | Val Acc: 0.7993 | Val F1-macro: 0.7988
Epoch 4 | Train Loss: 0.7109 | Val Loss: 0.7258 | Val Acc: 0.8172 | Val F1-macro: 0.8147
Epoch 5 | Train Loss: 0.6009 | Val Loss: 0.6485 | Val Acc: 0.8313 | Val F1-macro: 0.8321


In [ ]:
# 1. Инициализация предобученной модели Swin Transformer и настройка классификатора
model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.head = nn.Linear(model.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.head.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 184MB/s]


In [ ]:
# Обучение модели Swin Transformer
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.3414 | Val Loss: 1.6289 | Val Acc: 0.6179 | Val F1-macro: 0.6040
Epoch 2 | Train Loss: 1.4536 | Val Loss: 1.2238 | Val Acc: 0.7149 | Val F1-macro: 0.7126
Epoch 3 | Train Loss: 1.1613 | Val Loss: 1.0637 | Val Acc: 0.7358 | Val F1-macro: 0.7297
Epoch 4 | Train Loss: 0.9796 | Val Loss: 0.9316 | Val Acc: 0.7679 | Val F1-macro: 0.7672
Epoch 5 | Train Loss: 0.8609 | Val Loss: 0.8607 | Val Acc: 0.7896 | Val F1-macro: 0.7883


In [ ]:
# В ходе экспериментов было установлено, что альтернативные трансформерные модели не показали улучшения метрик качества по сравнению с Vision Transformer (ViT).
# Среди сверточных архитектур наилучшие результаты продемонстрировала модель ConvNeXt. В связи с этим для дальнейшего улучшения были выбраны модели Vision Transformer и ConvNeXt.

In [ ]:
# 2. Fine-tuning предобученной модели ViT: обучение всех слоев с малым learning rate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-5)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 148MB/s]


In [ ]:
# Обучение модели ViT в режиме fine-tuning
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.1985 | Val Loss: 1.3079 | Val Acc: 0.7522 | Val F1-macro: 0.7548
Epoch 2 | Train Loss: 0.8525 | Val Loss: 0.7032 | Val Acc: 0.8709 | Val F1-macro: 0.8694
Epoch 3 | Train Loss: 0.3990 | Val Loss: 0.4863 | Val Acc: 0.9112 | Val F1-macro: 0.9109
Epoch 4 | Train Loss: 0.1945 | Val Loss: 0.3821 | Val Acc: 0.9328 | Val F1-macro: 0.9315
Epoch 5 | Train Loss: 0.0964 | Val Loss: 0.3099 | Val Acc: 0.9351 | Val F1-macro: 0.9338


In [ ]:
# 2. Fine-tuning для ConvNeXt: обучение всех слоев с малым learning rate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
# Обучение модели ConvNeXt в режиме fine-tuning
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.0804 | Val Loss: 2.4138 | Val Acc: 0.5612 | Val F1-macro: 0.5459
Epoch 2 | Train Loss: 2.0802 | Val Loss: 1.6508 | Val Acc: 0.6940 | Val F1-macro: 0.6891
Epoch 3 | Train Loss: 1.4808 | Val Loss: 1.2231 | Val Acc: 0.7694 | Val F1-macro: 0.7671
Epoch 4 | Train Loss: 1.1088 | Val Loss: 0.9563 | Val Acc: 0.8254 | Val F1-macro: 0.8233
Epoch 5 | Train Loss: 0.8517 | Val Loss: 0.7621 | Val Acc: 0.8582 | Val F1-macro: 0.8569


In [ ]:
# В результате применения fine-tuning было получено улучшение значений метрик качества.
# Это подтверждает, что дообучение предобученых моделей позволило лучше адаптировать их к рассматриваемому датасету.

In [ ]:
# 3. Обучение модели ViT с новым значением learning rate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2276 | Val Loss: 0.5083 | Val Acc: 0.8799 | Val F1-macro: 0.8820
Epoch 2 | Train Loss: 0.1742 | Val Loss: 0.2667 | Val Acc: 0.9388 | Val F1-macro: 0.9386
Epoch 3 | Train Loss: 0.0262 | Val Loss: 0.1617 | Val Acc: 0.9604 | Val F1-macro: 0.9594
Epoch 4 | Train Loss: 0.0095 | Val Loss: 0.1405 | Val Acc: 0.9664 | Val F1-macro: 0.9654
Epoch 5 | Train Loss: 0.0046 | Val Loss: 0.1384 | Val Acc: 0.9649 | Val F1-macro: 0.9641


In [ ]:
# Обучение модели ViT с learning rate = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)


for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.1937 | Val Loss: 0.4051 | Val Acc: 0.9097 | Val F1-macro: 0.9117
Epoch 2 | Train Loss: 0.1467 | Val Loss: 0.2786 | Val Acc: 0.9321 | Val F1-macro: 0.9312
Epoch 3 | Train Loss: 0.0226 | Val Loss: 0.1728 | Val Acc: 0.9500 | Val F1-macro: 0.9495
Epoch 4 | Train Loss: 0.0040 | Val Loss: 0.1384 | Val Acc: 0.9619 | Val F1-macro: 0.9610
Epoch 5 | Train Loss: 0.0021 | Val Loss: 0.1282 | Val Acc: 0.9642 | Val F1-macro: 0.9630


In [ ]:
# 3. Обучение модели ConvNeXt с новым значением learning rate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.9211 | Val Loss: 0.8217 | Val Acc: 0.8500 | Val F1-macro: 0.8479
Epoch 2 | Train Loss: 0.5355 | Val Loss: 0.3281 | Val Acc: 0.9373 | Val F1-macro: 0.9365
Epoch 3 | Train Loss: 0.1908 | Val Loss: 0.2042 | Val Acc: 0.9493 | Val F1-macro: 0.9492
Epoch 4 | Train Loss: 0.0975 | Val Loss: 0.1444 | Val Acc: 0.9724 | Val F1-macro: 0.9715
Epoch 5 | Train Loss: 0.0459 | Val Loss: 0.1333 | Val Acc: 0.9627 | Val F1-macro: 0.9616


In [ ]:
# Обучение модели ConvNeXt с learning rate 1е-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.3907 | Val Loss: 0.3867 | Val Acc: 0.9201 | Val F1-macro: 0.9189
Epoch 2 | Train Loss: 0.2112 | Val Loss: 0.1881 | Val Acc: 0.9612 | Val F1-macro: 0.9600
Epoch 3 | Train Loss: 0.0485 | Val Loss: 0.1156 | Val Acc: 0.9739 | Val F1-macro: 0.9724
Epoch 4 | Train Loss: 0.0222 | Val Loss: 0.1181 | Val Acc: 0.9709 | Val F1-macro: 0.9697
Epoch 5 | Train Loss: 0.0139 | Val Loss: 0.1106 | Val Acc: 0.9687 | Val F1-macro: 0.9679


In [ ]:
# Обучение модели ConvNeXt с learning rate 5е-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.0857 | Val Loss: 0.3965 | Val Acc: 0.8940 | Val F1-macro: 0.8918
Epoch 2 | Train Loss: 0.1964 | Val Loss: 0.3489 | Val Acc: 0.9015 | Val F1-macro: 0.9020
Epoch 3 | Train Loss: 0.1243 | Val Loss: 0.2555 | Val Acc: 0.9343 | Val F1-macro: 0.9339
Epoch 4 | Train Loss: 0.0604 | Val Loss: 0.4841 | Val Acc: 0.8828 | Val F1-macro: 0.8884
Epoch 5 | Train Loss: 0.1964 | Val Loss: 0.3209 | Val Acc: 0.9127 | Val F1-macro: 0.9120


In [ ]:
# В ходе экспериментов с подбором learning rate было установлено,
# что значение 5e-5 обеспечивает наилучшие результаты для ViT, тогда как для ConvNeXt оптимальный learning rate равен 1e-4

In [ ]:
# 4. Применение аугментации изображений (горизонтальное отражение)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ViT с применением аугментации данных (горизонтальное отражение = 0.3)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2220 | Val Loss: 0.7922 | Val Acc: 0.7657 | Val F1-macro: 0.8229
Epoch 2 | Train Loss: 0.2294 | Val Loss: 0.2597 | Val Acc: 0.9418 | Val F1-macro: 0.9419
Epoch 3 | Train Loss: 0.0424 | Val Loss: 0.1789 | Val Acc: 0.9575 | Val F1-macro: 0.9570
Epoch 4 | Train Loss: 0.0148 | Val Loss: 0.1283 | Val Acc: 0.9716 | Val F1-macro: 0.9704
Epoch 5 | Train Loss: 0.0068 | Val Loss: 0.1245 | Val Acc: 0.9687 | Val F1-macro: 0.9674


In [ ]:
# Обучение модели ConvNeXt с применением аугментации данных (горизонтальное отражение = 0.3)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.4176 | Val Loss: 0.4274 | Val Acc: 0.9082 | Val F1-macro: 0.9082
Epoch 2 | Train Loss: 0.2672 | Val Loss: 0.1943 | Val Acc: 0.9530 | Val F1-macro: 0.9511
Epoch 3 | Train Loss: 0.0811 | Val Loss: 0.1240 | Val Acc: 0.9679 | Val F1-macro: 0.9667
Epoch 4 | Train Loss: 0.0344 | Val Loss: 0.1285 | Val Acc: 0.9649 | Val F1-macro: 0.9623
Epoch 5 | Train Loss: 0.0255 | Val Loss: 0.0896 | Val Acc: 0.9746 | Val F1-macro: 0.9734


In [ ]:
# Применение аугментации данных (горизонтальное отражение = 0.5)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ViT с применением аугментации данных (горизонтальное отражение = 0.5)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2501 | Val Loss: 0.7807 | Val Acc: 0.7739 | Val F1-macro: 0.8247
Epoch 2 | Train Loss: 0.2469 | Val Loss: 0.2948 | Val Acc: 0.9254 | Val F1-macro: 0.9252
Epoch 3 | Train Loss: 0.0469 | Val Loss: 0.2219 | Val Acc: 0.9366 | Val F1-macro: 0.9362
Epoch 4 | Train Loss: 0.0197 | Val Loss: 0.1498 | Val Acc: 0.9634 | Val F1-macro: 0.9627
Epoch 5 | Train Loss: 0.0070 | Val Loss: 0.1196 | Val Acc: 0.9694 | Val F1-macro: 0.9684


In [ ]:
# Обучение модели ConvNeXt с применением аугментации данных (горизонтальное отражение = 0.5)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.4279 | Val Loss: 0.4326 | Val Acc: 0.9022 | Val F1-macro: 0.9045
Epoch 2 | Train Loss: 0.2505 | Val Loss: 0.1526 | Val Acc: 0.9687 | Val F1-macro: 0.9675
Epoch 3 | Train Loss: 0.0790 | Val Loss: 0.1249 | Val Acc: 0.9694 | Val F1-macro: 0.9678
Epoch 4 | Train Loss: 0.0338 | Val Loss: 0.1001 | Val Acc: 0.9687 | Val F1-macro: 0.9669
Epoch 5 | Train Loss: 0.0188 | Val Loss: 0.0971 | Val Acc: 0.9739 | Val F1-macro: 0.9725


In [ ]:
# Лучший результат получила конфигурация с горизонтальным отражением = 0.3

In [ ]:
# 4. Применение аугментации данных (горизонтальное отражение и небольшое вращение)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ViT с применением аугментации данных (горизонтальное отражение и небольшое вращение)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.3261 | Val Loss: 0.8028 | Val Acc: 0.7776 | Val F1-macro: 0.8190
Epoch 2 | Train Loss: 0.3143 | Val Loss: 0.2825 | Val Acc: 0.9366 | Val F1-macro: 0.9371
Epoch 3 | Train Loss: 0.0947 | Val Loss: 0.1956 | Val Acc: 0.9545 | Val F1-macro: 0.9543
Epoch 4 | Train Loss: 0.0415 | Val Loss: 0.1604 | Val Acc: 0.9604 | Val F1-macro: 0.9610
Epoch 5 | Train Loss: 0.0239 | Val Loss: 0.1720 | Val Acc: 0.9575 | Val F1-macro: 0.9570


In [ ]:
# Обучение модели ConvNeXt с применением аугментации данных (горизонтальное отражение и небольшое вращение)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.5339 | Val Loss: 0.5740 | Val Acc: 0.8604 | Val F1-macro: 0.8677
Epoch 2 | Train Loss: 0.3300 | Val Loss: 0.2112 | Val Acc: 0.9485 | Val F1-macro: 0.9470
Epoch 3 | Train Loss: 0.1090 | Val Loss: 0.1291 | Val Acc: 0.9664 | Val F1-macro: 0.9660
Epoch 4 | Train Loss: 0.0486 | Val Loss: 0.1279 | Val Acc: 0.9634 | Val F1-macro: 0.9627
Epoch 5 | Train Loss: 0.0330 | Val Loss: 0.0890 | Val Acc: 0.9769 | Val F1-macro: 0.9754


In [ ]:
# Вращение не улучшило значения метрик качества для ViT. В случае с ConvNeXt вращение немного улучшило метрики

In [ ]:
# 4. Добавление аугментации (яркость и контраст)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ViT с применением аугментации данных (горизонтальное отражение, яркость и контраст)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2915 | Val Loss: 0.6427 | Val Acc: 0.8239 | Val F1-macro: 0.8535
Epoch 2 | Train Loss: 0.2581 | Val Loss: 0.2773 | Val Acc: 0.9343 | Val F1-macro: 0.9332
Epoch 3 | Train Loss: 0.0475 | Val Loss: 0.1723 | Val Acc: 0.9575 | Val F1-macro: 0.9569
Epoch 4 | Train Loss: 0.0154 | Val Loss: 0.1273 | Val Acc: 0.9657 | Val F1-macro: 0.9648
Epoch 5 | Train Loss: 0.0063 | Val Loss: 0.1223 | Val Acc: 0.9657 | Val F1-macro: 0.9654


In [ ]:
# Добавление аугментации (яркость и контраст, новые значения)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ViT с применением аугментации данных (горизонтальное отражение, а также яркость и контраст с новыми значениями)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2645 | Val Loss: 0.5834 | Val Acc: 0.8440 | Val F1-macro: 0.8690
Epoch 2 | Train Loss: 0.2421 | Val Loss: 0.2792 | Val Acc: 0.9321 | Val F1-macro: 0.9322
Epoch 3 | Train Loss: 0.0477 | Val Loss: 0.1797 | Val Acc: 0.9552 | Val F1-macro: 0.9548
Epoch 4 | Train Loss: 0.0144 | Val Loss: 0.1160 | Val Acc: 0.9716 | Val F1-macro: 0.9705
Epoch 5 | Train Loss: 0.0062 | Val Loss: 0.1090 | Val Acc: 0.9754 | Val F1-macro: 0.9740


In [ ]:
# Добавление аугментации (яркость и контраст, горизонтальное отражение и небольшое вращение)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Обучение модели ConvNeXt с применением аугментации данных (горизонтальное отражение и небольшое вращение, а также яркость и контраст)
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.5152 | Val Loss: 0.4413 | Val Acc: 0.8940 | Val F1-macro: 0.8934
Epoch 2 | Train Loss: 0.2903 | Val Loss: 0.2024 | Val Acc: 0.9515 | Val F1-macro: 0.9499
Epoch 3 | Train Loss: 0.1013 | Val Loss: 0.1181 | Val Acc: 0.9687 | Val F1-macro: 0.9671
Epoch 4 | Train Loss: 0.0464 | Val Loss: 0.1135 | Val Acc: 0.9672 | Val F1-macro: 0.9658
Epoch 5 | Train Loss: 0.0337 | Val Loss: 0.1008 | Val Acc: 0.9746 | Val F1-macro: 0.9741


In [ ]:
# Установлено, что более интенсивное изменение яркости и контраста не улучшает качество,
# тогда как умеренные изменения позволяют достичь лучших значений метрик - для модели ViT
# Для ConvNeXt установлено, что яркость и контраст не способствуют улучшению качества

In [ ]:
# В ходе экспериментов с аугментациями установлено, что горизонтальное отражение улучшает качество,
# тогда как добавление вращения приводит к его снижению для трасформерной модели (ViT) и к дополнительному повышению для сверточной (ConvNeXt).
# Также показано, что умеренные изменения яркости и контраста дополнительно повышают значения метрик для трансформерной модели (ViT), в то время как более интенсивные
# преобразования не дают прироста качества. Для сверточной модели (ConvNeXt) яркость и контраст в принципе не являются эффективным инструментом повышения качества метрик

In [ ]:
# 5. Обучение модели ViT с увеличенным batch size
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.5033 | Val Loss: 0.5500 | Val Acc: 0.8970 | Val F1-macro: 0.8967
Epoch 2 | Train Loss: 0.2680 | Val Loss: 0.2964 | Val Acc: 0.9381 | Val F1-macro: 0.9391
Epoch 3 | Train Loss: 0.0633 | Val Loss: 0.1870 | Val Acc: 0.9664 | Val F1-macro: 0.9653
Epoch 4 | Train Loss: 0.0187 | Val Loss: 0.1364 | Val Acc: 0.9724 | Val F1-macro: 0.9713
Epoch 5 | Train Loss: 0.0093 | Val Loss: 0.1221 | Val Acc: 0.9709 | Val F1-macro: 0.9696


In [ ]:
# Обучение модели ViT с уменьшенным batch size
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.1969 | Val Loss: 0.5643 | Val Acc: 0.8381 | Val F1-macro: 0.8618
Epoch 2 | Train Loss: 0.1867 | Val Loss: 0.2587 | Val Acc: 0.9351 | Val F1-macro: 0.9326
Epoch 3 | Train Loss: 0.0369 | Val Loss: 0.2171 | Val Acc: 0.9455 | Val F1-macro: 0.9426
Epoch 4 | Train Loss: 0.0119 | Val Loss: 0.2038 | Val Acc: 0.9493 | Val F1-macro: 0.9503
Epoch 5 | Train Loss: 0.0049 | Val Loss: 0.1036 | Val Acc: 0.9776 | Val F1-macro: 0.9771


In [ ]:
# Обучение модели ViT с batch size = 8
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.0465 | Val Loss: 0.3972 | Val Acc: 0.8888 | Val F1-macro: 0.8901
Epoch 2 | Train Loss: 0.1579 | Val Loss: 0.2866 | Val Acc: 0.9119 | Val F1-macro: 0.9124
Epoch 3 | Train Loss: 0.0492 | Val Loss: 0.2417 | Val Acc: 0.9321 | Val F1-macro: 0.9324
Epoch 4 | Train Loss: 0.0671 | Val Loss: 0.4107 | Val Acc: 0.8888 | Val F1-macro: 0.8884
Epoch 5 | Train Loss: 0.0784 | Val Loss: 0.2441 | Val Acc: 0.9313 | Val F1-macro: 0.9303


In [ ]:
# 5. Обучение модели ConvNeXt с увеличенным batch size
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.9161 | Val Loss: 0.7097 | Val Acc: 0.8597 | Val F1-macro: 0.8580
Epoch 2 | Train Loss: 0.4504 | Val Loss: 0.2502 | Val Acc: 0.9493 | Val F1-macro: 0.9481
Epoch 3 | Train Loss: 0.1581 | Val Loss: 0.1668 | Val Acc: 0.9575 | Val F1-macro: 0.9579
Epoch 4 | Train Loss: 0.0724 | Val Loss: 0.1155 | Val Acc: 0.9679 | Val F1-macro: 0.9673
Epoch 5 | Train Loss: 0.0415 | Val Loss: 0.0937 | Val Acc: 0.9739 | Val F1-macro: 0.9733


In [ ]:
# Обучение модели ConvNeXt с уменьшенным batch size
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.2317 | Val Loss: 0.3278 | Val Acc: 0.9216 | Val F1-macro: 0.9233
Epoch 2 | Train Loss: 0.2099 | Val Loss: 0.1584 | Val Acc: 0.9575 | Val F1-macro: 0.9561
Epoch 3 | Train Loss: 0.0649 | Val Loss: 0.1004 | Val Acc: 0.9791 | Val F1-macro: 0.9786
Epoch 4 | Train Loss: 0.0358 | Val Loss: 0.0880 | Val Acc: 0.9813 | Val F1-macro: 0.9808
Epoch 5 | Train Loss: 0.0297 | Val Loss: 0.1061 | Val Acc: 0.9754 | Val F1-macro: 0.9741


In [ ]:
# Обучение модели ConvNeXt с batch size = 8
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 1.0733 | Val Loss: 0.3019 | Val Acc: 0.9201 | Val F1-macro: 0.9215
Epoch 2 | Train Loss: 0.1729 | Val Loss: 0.1572 | Val Acc: 0.9575 | Val F1-macro: 0.9561
Epoch 3 | Train Loss: 0.0731 | Val Loss: 0.1287 | Val Acc: 0.9679 | Val F1-macro: 0.9669
Epoch 4 | Train Loss: 0.0619 | Val Loss: 0.1571 | Val Acc: 0.9604 | Val F1-macro: 0.9585
Epoch 5 | Train Loss: 0.0478 | Val Loss: 0.1252 | Val Acc: 0.9694 | Val F1-macro: 0.9674


In [ ]:
# В ходе работы установлено, что batch size = 16 обеспечивает наилучшие значения метрик качества.
# При этом увеличение до 64 приводит к ухудшению результатов, а уменьшение до 8 также уступает по качеству

In [ ]:
# 6. Обучение модели ViT с целью определения оптимального количества эпох
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

best_f1 = 0.0
best_model_state = None

for epoch in range(8):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

_ = model.load_state_dict(best_model_state)

Epoch 1 | Train Loss: 1.0830 | Val Loss: 0.6114 | Val Acc: 0.8358 | Val F1-macro: 0.8565
Epoch 2 | Train Loss: 0.1734 | Val Loss: 0.2896 | Val Acc: 0.9172 | Val F1-macro: 0.9186
Epoch 3 | Train Loss: 0.0287 | Val Loss: 0.1416 | Val Acc: 0.9672 | Val F1-macro: 0.9661
Epoch 4 | Train Loss: 0.0087 | Val Loss: 0.1047 | Val Acc: 0.9761 | Val F1-macro: 0.9754
Epoch 5 | Train Loss: 0.0030 | Val Loss: 0.1021 | Val Acc: 0.9791 | Val F1-macro: 0.9779
Epoch 6 | Train Loss: 0.0020 | Val Loss: 0.0991 | Val Acc: 0.9791 | Val F1-macro: 0.9781
Epoch 7 | Train Loss: 0.0015 | Val Loss: 0.0975 | Val Acc: 0.9784 | Val F1-macro: 0.9774
Epoch 8 | Train Loss: 0.0011 | Val Loss: 0.1005 | Val Acc: 0.9784 | Val F1-macro: 0.9774


In [ ]:
# В ходе эксперимента установлено, что в случае с ViT наилучшее качество достигается на 5-й и 6-й эпохах обучения, однако обучение имеет смысл вплоть до седьмой-восьмой эпохи
# В случае с ConvNeXt наилучшее качество достигается на 3-4 эпохе

In [ ]:
# Оценка модели ViT на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.9575 | Test F1-macro: 0.9563


In [ ]:
# Оценка модели ConvNeXt на тестовой выборке с предварительным обучением улучшенной финальной модели
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(4):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 134MB/s] 


Test Acc: 0.9530 | Test F1-macro: 0.9509


In [ ]:
# Имплементация модели ResNet18
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # skip connection
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = F.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7,
                               stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = []

        # первый блок (может менять размер)
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

def MyResNet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

In [ ]:
# Имплементация модели Vision Transformer
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=256):
        super().__init__()

        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, E = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = attn @ v
        x = x.transpose(1, 2).reshape(B, N, E)

        x = self.proj(x)
        return x

class MLP(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class MyViT(nn.Module):
    def __init__(self, num_classes, embed_dim=256, depth=6, num_heads=8):
        super().__init__()

        self.patch_embed = PatchEmbedding(embed_dim=embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)

        x = x + self.pos_embed

        x = self.blocks(x)
        x = self.norm(x)

        cls_output = x[:, 0]  # берем только class token
        out = self.head(cls_output)

        return out

In [ ]:
# Обучение baseline-модели MyResNet18
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyResNet18(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 2.9683 | Val Loss: 4.0285 | Val Acc: 0.1709 | Val F1-macro: 0.1192
Epoch 2 | Train Loss: 2.3075 | Val Loss: 3.4650 | Val Acc: 0.1694 | Val F1-macro: 0.1276
Epoch 3 | Train Loss: 1.8906 | Val Loss: 3.6585 | Val Acc: 0.2836 | Val F1-macro: 0.2425
Epoch 4 | Train Loss: 1.6649 | Val Loss: 2.3250 | Val Acc: 0.3575 | Val F1-macro: 0.3228
Epoch 5 | Train Loss: 1.4876 | Val Loss: 1.5088 | Val Acc: 0.5836 | Val F1-macro: 0.5879


In [ ]:
# Оценка модели на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.5791 | Test F1-macro: 0.5859


In [ ]:
# Обучение baseline-модели MyViT
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyViT(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.2174 | Val Loss: 3.0633 | Val Acc: 0.1545 | Val F1-macro: 0.1078
Epoch 2 | Train Loss: 2.8733 | Val Loss: 2.9265 | Val Acc: 0.1784 | Val F1-macro: 0.1256
Epoch 3 | Train Loss: 2.6964 | Val Loss: 3.0714 | Val Acc: 0.1776 | Val F1-macro: 0.1445
Epoch 4 | Train Loss: 2.4424 | Val Loss: 2.5663 | Val Acc: 0.2485 | Val F1-macro: 0.2205
Epoch 5 | Train Loss: 2.2279 | Val Loss: 2.3745 | Val Acc: 0.3291 | Val F1-macro: 0.3248


In [ ]:
# Оценка модели на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.3261 | Test F1-macro: 0.3218


In [ ]:
# Имплементация модели ConvNeXt
class LayerNorm2d(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)
        return x

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, layer_scale_init_value=1e-6):
        super().__init__()

        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)

        if layer_scale_init_value > 0:
            self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim))
        else:
            self.gamma = None

    def forward(self, x):
        identity = x

        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)

        if self.gamma is not None:
            x = self.gamma.view(1, -1, 1, 1) * x

        x = x + identity
        return x


class DownsampleLayer(nn.Module):
    def __init__(self, in_dim, out_dim, stride=2):
        super().__init__()
        self.norm = LayerNorm2d(in_dim)
        self.reduction = nn.Conv2d(in_dim, out_dim, kernel_size=stride, stride=stride)

    def forward(self, x):
        x = self.norm(x)
        x = self.reduction(x)
        return x


class ConvNeXt(nn.Module):
    def __init__(self, num_classes=10, depths=(2, 2, 2, 2), dims=(96, 192, 384, 768)):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            LayerNorm2d(dims[0])
        )

        self.stage1 = nn.Sequential(*[ConvNeXtBlock(dims[0]) for _ in range(depths[0])])
        self.down1 = DownsampleLayer(dims[0], dims[1])

        self.stage2 = nn.Sequential(*[ConvNeXtBlock(dims[1]) for _ in range(depths[1])])
        self.down2 = DownsampleLayer(dims[1], dims[2])

        self.stage3 = nn.Sequential(*[ConvNeXtBlock(dims[2]) for _ in range(depths[2])])
        self.down3 = DownsampleLayer(dims[2], dims[3])

        self.stage4 = nn.Sequential(*[ConvNeXtBlock(dims[3]) for _ in range(depths[3])])

        self.norm = nn.LayerNorm(dims[3])
        self.head = nn.Linear(dims[3], num_classes)

    def forward(self, x):
        x = self.stem(x)

        x = self.stage1(x)
        x = self.down1(x)

        x = self.stage2(x)
        x = self.down2(x)

        x = self.stage3(x)
        x = self.down3(x)

        x = self.stage4(x)

        x = x.mean(dim=[2, 3])
        x = self.norm(x)
        x = self.head(x)
        return x


def ConvNeXtTinyCustom(num_classes):
    return ConvNeXt(
        num_classes=num_classes,
        depths=(2, 2, 2, 2),
        dims=(96, 192, 384, 768)
    )

In [ ]:
# Добавление техник улучшенного бейзлайна и последующее обучение (ConvNeXt)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvNeXtTinyCustom(num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.3028 | Val Loss: 3.1390 | Val Acc: 0.1187 | Val F1-macro: 0.0636
Epoch 2 | Train Loss: 2.8858 | Val Loss: 2.6702 | Val Acc: 0.2403 | Val F1-macro: 0.1952
Epoch 3 | Train Loss: 2.4843 | Val Loss: 2.4682 | Val Acc: 0.2694 | Val F1-macro: 0.2378
Epoch 4 | Train Loss: 2.2377 | Val Loss: 2.1796 | Val Acc: 0.3851 | Val F1-macro: 0.3593
Epoch 5 | Train Loss: 2.0382 | Val Loss: 2.0481 | Val Acc: 0.4261 | Val F1-macro: 0.4163


In [ ]:
# Оценка модели на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.4209 | Test F1-macro: 0.4138


In [ ]:
# Добавление техник улучшенного бейзлайна и последующее обучение с большим количеством эпох (ConvNeXt)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvNeXtTinyCustom(num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(9):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.3154 | Val Loss: 3.1732 | Val Acc: 0.1067 | Val F1-macro: 0.0606
Epoch 2 | Train Loss: 2.9332 | Val Loss: 2.7179 | Val Acc: 0.2425 | Val F1-macro: 0.1959
Epoch 3 | Train Loss: 2.4978 | Val Loss: 2.4218 | Val Acc: 0.2851 | Val F1-macro: 0.2605
Epoch 4 | Train Loss: 2.2449 | Val Loss: 2.1996 | Val Acc: 0.3694 | Val F1-macro: 0.3394
Epoch 5 | Train Loss: 2.0375 | Val Loss: 2.0616 | Val Acc: 0.4164 | Val F1-macro: 0.4096
Epoch 6 | Train Loss: 1.8784 | Val Loss: 1.9119 | Val Acc: 0.4866 | Val F1-macro: 0.4895
Epoch 7 | Train Loss: 1.8046 | Val Loss: 1.9024 | Val Acc: 0.4799 | Val F1-macro: 0.5059
Epoch 8 | Train Loss: 1.7125 | Val Loss: 1.8145 | Val Acc: 0.5022 | Val F1-macro: 0.5202
Epoch 9 | Train Loss: 1.6732 | Val Loss: 1.7293 | Val Acc: 0.5343 | Val F1-macro: 0.5394


In [ ]:
# Оценка модели на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.5321 | Test F1-macro: 0.5445


In [ ]:
# Добавление техник улучшенного бейзлайна и последующее обучение (ViT)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyViT(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.0519 | Val Loss: 2.8145 | Val Acc: 0.2022 | Val F1-macro: 0.1594
Epoch 2 | Train Loss: 2.6425 | Val Loss: 2.4285 | Val Acc: 0.2940 | Val F1-macro: 0.2491
Epoch 3 | Train Loss: 2.3558 | Val Loss: 2.1808 | Val Acc: 0.3776 | Val F1-macro: 0.3620
Epoch 4 | Train Loss: 2.1047 | Val Loss: 1.9991 | Val Acc: 0.4478 | Val F1-macro: 0.4425
Epoch 5 | Train Loss: 1.9229 | Val Loss: 1.8960 | Val Acc: 0.4672 | Val F1-macro: 0.4680


In [ ]:
# Оценка модели на тестовой выборке (улучшенный бейзлайн)
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.4582 | Test F1-macro: 0.4532


In [ ]:
# Добавление техник улучшенного бейзлайна и последующее обучение с большим количеством эпох (ViT)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyViT(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

for epoch in range(8):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    epoch_loss = total_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

Epoch 1 | Train Loss: 3.0656 | Val Loss: 2.8376 | Val Acc: 0.1925 | Val F1-macro: 0.1456
Epoch 2 | Train Loss: 2.6726 | Val Loss: 2.4897 | Val Acc: 0.2873 | Val F1-macro: 0.2237
Epoch 3 | Train Loss: 2.3957 | Val Loss: 2.2158 | Val Acc: 0.3836 | Val F1-macro: 0.3659
Epoch 4 | Train Loss: 2.1432 | Val Loss: 2.0976 | Val Acc: 0.3940 | Val F1-macro: 0.3689
Epoch 5 | Train Loss: 1.9564 | Val Loss: 2.0116 | Val Acc: 0.4172 | Val F1-macro: 0.4118
Epoch 6 | Train Loss: 1.8215 | Val Loss: 1.8203 | Val Acc: 0.4873 | Val F1-macro: 0.4768
Epoch 7 | Train Loss: 1.6877 | Val Loss: 1.8020 | Val Acc: 0.4821 | Val F1-macro: 0.4773
Epoch 8 | Train Loss: 1.5941 | Val Loss: 1.8193 | Val Acc: 0.4925 | Val F1-macro: 0.4895


In [ ]:
# Оценка модели на тестовой выборке
model.eval()

correct = 0
total = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = correct / total
test_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Test Acc: {test_acc:.4f} | Test F1-macro: {test_f1:.4f}")

Test Acc: 0.4716 | Test F1-macro: 0.4645
